# LLM Fine-Tuning Deep Dive, Part 2 of 3: Parameter-Based Techniques + TF Quantization (TensorFlow/Keras)

> **This is Part 2 of a three-notebook fine-tuning arc:**
>
> 1. [Part 1: Data-based techniques](01-llm-finetuning-data-techniques.ipynb) — continued pretraining (full FT), instruction tuning (layer freezing), preference-signal FT. Trains and saves three checkpoints.
> 2. **Part 2 (this notebook): Parameter-based techniques + TF quantization** — how many/which weights are updated (full fine-tuning, partial freezing, TF-native parameter-efficient FT), then TFLite post-training quantization.
> 3. [Part 3: Comparison & decision](03-llm-finetuning-comparison-and-decision.ipynb) — head-to-head evaluation of all trained checkpoints, held-out perplexity, ablation study, and the final call.

**Recap of where Part 1 left off:** Riverside now has a model that knows its catalog (`non-instruction-full`), follows instructions (`instruction-ft`), and has been preference-aligned (`preference-ft`). That's the *data axis*. Part 1 made implicit parameter choices for each stage; this notebook makes the parameter choice the *variable*: we use continued pretraining as the controlled test bed — same data throughout, so any output difference comes only from how many weights updated.

## Table of Contents (Part 2)

1. [Setup: Reloading Where Part 1 Left Off](#setup-reloading-where-part-1-left-off)
2. [Parameter-Based Axis: How Many Weights Do We Actually Update?](#parameter-based-axis)
   - [Concept 4: Full Fine-Tuning](#concept-4-full-fine-tuning)
   - [Concept 5: Partial Freezing](#concept-5-partial-fine-tuning-layer-freezing)
   - [Concept 6: TF-Native Parameter-Efficient Fine-Tuning](#concept-6-tf-native-parameter-efficient-fine-tuning)
3. [TF Note: LoRA and QLoRA (PyTorch-only — concept walkthrough)](#tf-note-lora-and-qlora)
4. [Quantization for Deployment: TFLite Post-Training Quantization](#quantization-for-deployment)
5. [Visual Comparison: Parameter Counts Across All Techniques](#visual-comparison)

---

## Parameter Strategies at a Glance

These methods differ in which parameters receive updates and how much optimizer-state memory they require:

| Technique | Trainable % | Memory | Quality | Forgetting Risk | When to Use |
| --- | --- | --- | --- | --- | --- |
| **Full fine-tuning** | 100% | Highest | Highest | Highest | Small models, abundant compute |
| **Partial freezing** | 10–30% | Medium | Medium–High | Medium | Limited compute budget |
| **TF parameter-efficient FT** | <5% | Lowest | High | Lowest | Most production scenarios |
| **LoRA (PyTorch)** | <1% | Lowest | High | Lowest | Production with PyTorch stack |

### The Two Axes Are Complementary, Not Either/Or

| Data objective ↓ / Parameter strategy → | **Full FT (100%)** | **Partial Freeze (~21%)** | **Parameter-Efficient (<5%)** |
| --- | --- | --- | --- |
| **Continued Pretraining** | Trained (Part 1: `non-instruction-full`) | Trained (this notebook: `partial-freeze`) | Trained (this notebook: `peft-tf`) |
| **Instruction Tuning (SFT)** | Not trained | Not trained | Trained (Part 1: `instruction-ft` via layer freezing) |
| **Preference FT** | Not trained | Not trained | Trained (Part 1: `preference-ft`) |

The three continued-pretraining checkpoints form the cleanest controlled comparison: same data, same objective, only the parameter budget differs.

In [ ]:
# Re-establishing Part 1's foundations.
import subprocess, sys, warnings
required = [
    ('numpy', 'numpy'), ('matplotlib', 'matplotlib'),
    ('tensorflow', 'tensorflow'), ('transformers', 'transformers'),
]
for imp, pkg in required:
    try:
        __import__(imp)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import tensorflow as tf
from transformers import TFGPT2LMHeadModel, GPT2Tokenizer

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

MODEL_NAME = 'gpt2-medium'
PROMPT = 'Aria Voss stared at the signal counting itself out in prime numbers and'
INSTRUCTION_PREFIX = 'Continue the fiction narrative in the same style:\n\n'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)
total_params = sum(np.prod(v.shape) for v in base_model.variables)


def generate(model, prompt, max_new_tokens=60):
    input_ids = tokenizer(prompt, return_tensors='tf')['input_ids']
    prompt_len = input_ids.shape[1]
    output = model.generate(
        input_ids, max_new_tokens=max_new_tokens,
        do_sample=True, top_p=0.9, temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
    )
    decoded = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()
    return decoded if decoded else '[model stopped — EOS as first token]'


print(f'Setup complete. Total params: {total_params / 1e6:.1f}M')
print(f'Baseline: {generate(base_model, PROMPT, max_new_tokens=30)}')

In [ ]:
# Corpus loader (identical to Part 1)
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / 'content'
if not CONTENT_DIR.exists():
    _fallback = Path.cwd() / 'learning' / 'genai' / '04-llm' / 'content'
    if _fallback.exists():
        CONTENT_DIR = _fallback

NOVELS = {
    'scifi': 'the-weight-of-distant-light',
    'fantasy': 'the-tidebound-accord',
    'mystery': 'the-cartographers-cipher',
    'historical': 'the-silk-merchants-daughter',
    'cyberpunk': 'neural-drift',
    'horror': 'the-hollow-beneath',
    'literary': 'the-weight-of-tides',
}


def load_corpus_paragraphs(novels=None, max_chapters=10, min_len=200):
    if novels is None:
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():
            continue
        chapter_files = sorted(novel_path.glob('chapter-*.txt'))[:max_chapters]
        for path in chapter_files:
            text = path.read_text(encoding='utf-8')
            for para in text.split('\n\n'):
                para = para.strip().replace('\n', ' ')
                if len(para) >= min_len:
                    paragraphs.append(para)
    return paragraphs


def tokenize_batch(paragraphs, max_length=128):
    enc = tokenizer(
        paragraphs, return_tensors='tf', max_length=max_length,
        padding='max_length', truncation=True,
    )
    return enc['input_ids'], enc['attention_mask']


print(f'Content directory: {CONTENT_DIR.absolute()}')

### Reloading Part 1's Checkpoints

Kernels don't share memory between notebooks. This section reloads the instruction-tuned checkpoint from `./checkpoints/instruction-ft` so the later LoRA discussion has a real trained adapter to inspect conceptually.

In [ ]:
import os

# Reload Part 1's continued-pretraining checkpoint (the full fine-tune)
non_instruct_ckpt_path = './checkpoints/non-instruction-full'
if os.path.isdir(non_instruct_ckpt_path):
    non_instruct_ckpt = TFGPT2LMHeadModel.from_pretrained(non_instruct_ckpt_path)
    print(f'Reloaded: {non_instruct_ckpt_path}')
    print(f'  Sanity check: {generate(non_instruct_ckpt, PROMPT, max_new_tokens=30)}')
else:
    print(f'WARNING: {non_instruct_ckpt_path} not found. Run Part 1 first.')
    non_instruct_ckpt = None

# Reload instruction-tuned checkpoint
instruct_ckpt_path = './checkpoints/instruction-ft'
if os.path.isdir(instruct_ckpt_path):
    instruct_model_reloaded = TFGPT2LMHeadModel.from_pretrained(instruct_ckpt_path)
    print(f'Reloaded: {instruct_ckpt_path}')
    print(f'  Sanity check: {generate(instruct_model_reloaded, INSTRUCTION_PREFIX + PROMPT + chr(10)*2, max_new_tokens=30)}')
else:
    print(f'WARNING: {instruct_ckpt_path} not found. Run Part 1 first.')
    instruct_model_reloaded = None

---

## Parameter-Based Axis: How Many Weights Do We Actually Update?

**Riverside's question for this section:** IT has given us one laptop CPU and a deadline. We now have a model that knows the lore, follows instructions, and matches editorial taste — but which of the three ways to *train* it can we actually afford to run and re-run as the catalog grows?

### The Cost Problem

All three data-based techniques (continued pretraining, instruction tuning, preference alignment) work by gradient descent on model weights. But **updating all weights is expensive:**

- **Memory:** ~355M parameters × (4 bytes/param + 8 bytes optimizer state) ≈ 4.3 GB just for `gpt2-medium`. For 70B models: **840 GB**.
- **Compute:** More trainable params = longer training time.
- **Risk:** Full updates can "overwrite" the model's general knowledge (catastrophic forgetting).

---

## Concept 4 (Parameter-Based): Full Fine-Tuning

**What it is:** Every single weight in the model is unfrozen and updated by the optimizer.

**Pros:** The model has maximum "room" to adapt to Riverside's catalog.

**Cons:** Highest memory/compute cost; highest forgetting risk on small corpora.

We already ran this in Part 1's Concept 1 (`./checkpoints/non-instruction-full`). The cell below quantifies what "100% trainable" looks like for `gpt2-medium`.

In [ ]:
# Quantify full fine-tuning parameter count for gpt2-medium
param_check = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)
total = sum(np.prod(v.shape) for v in param_check.variables)
trainable = sum(np.prod(v.shape) for v in param_check.trainable_variables)
print(f'Full fine-tuning: {trainable:,} / {total:,} parameters trainable ({trainable / total * 100:.1f}%)')
print(f'Optimizer state (Adam): ~{trainable * 8 / 1e9:.2f} GB  (2 momentum tensors × 4 bytes)')
print(f'Weights alone:          ~{trainable * 4 / 1e9:.2f} GB  (float32)')
print(f'Total training memory:  ~{trainable * 12 / 1e9:.2f} GB  (weights + optimizer state)')
del param_check

---

## Concept 5 (Parameter-Based): Partial Fine-Tuning (Layer Freezing)

**Riverside's question for this section:** if full fine-tuning is the "maximum quality, maximum laptop-fan-noise" option, is there a middle ground that still lets us re-train every time the catalog grows, without waiting hours?

**The observation:** In transformer models, **early layers** learn general language features (tokenization, basic syntax, common words) while **later layers** learn task-specific patterns.

**The strategy:** Freeze everything, then selectively unfreeze:
- The last N transformer blocks (task-specific adaptation)
- The final layer norm + output head (vocabulary projection)

**Pros:** Much cheaper than full fine-tuning; less prone to catastrophic forgetting.

**Cons:** Still edits raw model weights (can't "swap" like an adapter); choosing how many layers to unfreeze is a manual hyperparameter.

**Configuration for `gpt2-medium` (24 transformer blocks):** unfreeze only the last ~25% of blocks (6 blocks) + output head.

### Visualizing Layer-by-Layer Freezing

| Layer group | What it learns | Freeze? |
| --- | --- | --- |
| Early blocks (~first 50%) | Basic syntax, common words, tokenization | FROZEN |
| Middle blocks (~50–75%) | Mid-level semantics, phrase structure | FROZEN |
| Last ~25% of blocks | Task-specific / domain adaptation | TRAINABLE |
| Output head | Final vocabulary distribution | TRAINABLE |

In [ ]:
# Visualize partial freezing on gpt2-medium's 24 transformer blocks
n_layers = 24  # gpt2-medium
unfreeze_from = n_layers - max(2, n_layers // 4)  # 18: unfreeze last 6
layers = [f'Block {i}' for i in range(n_layers)] + ['Output Head']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, max(5, n_layers * 0.28)))

# Frozen vs Trainable
colors_pt = ['#aec6e8' if i < unfreeze_from else '#f4a261' for i in range(n_layers)] + ['#f4a261']
ax1.barh(range(len(layers)), [1] * len(layers), color=colors_pt, edgecolor='black', linewidth=0.8)
ax1.set_yticks(range(0, len(layers), 4))
ax1.set_yticklabels([layers[i] for i in range(0, len(layers), 4)], fontsize=8)
ax1.set_xlim(0, 1); ax1.set_xticks([])
ax1.set_title(f'Partial Fine-Tuning Strategy\n(gpt2-medium: {n_layers} blocks)', fontsize=11, fontweight='bold')
ax1.invert_yaxis()
ax1.legend(
    handles=[mpatches.Patch(facecolor='#aec6e8', edgecolor='black', label='FROZEN'),
             mpatches.Patch(facecolor='#f4a261', edgecolor='black', label='TRAINABLE')],
    loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=2, fontsize=9
)

# Gradient flow magnitude
grad_flow = [0.0] * unfreeze_from + list(np.linspace(0.4, 1.0, n_layers - unfreeze_from)) + [1.0]
ax2.barh(range(len(layers)), grad_flow, color='#52b788', alpha=0.8, edgecolor='black', linewidth=0.8)
ax2.set_yticks(range(0, len(layers), 4))
ax2.set_yticklabels([layers[i] for i in range(0, len(layers), 4)], fontsize=8)
ax2.set_xlabel('Relative gradient magnitude', fontsize=9)
ax2.set_title('Gradient Flow During Backpropagation', fontsize=11, fontweight='bold')
ax2.invert_yaxis()
ax2.text(0.05, unfreeze_from / 2, 'No gradient\n(frozen)', ha='center', va='center', fontsize=8, color='gray', style='italic')
ax2.text(0.7, unfreeze_from + (n_layers - unfreeze_from) / 2, 'Full gradient', ha='center', va='center', fontsize=8, color='#1b4332', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Configuration: freeze blocks 0-{unfreeze_from - 1}, train blocks {unfreeze_from}-{n_layers - 1} + output head')
print(f'Frozen blocks:    {unfreeze_from}/{n_layers} ({unfreeze_from / n_layers * 100:.1f}%)')
print(f'Trainable blocks: {n_layers - unfreeze_from}/{n_layers} ({(n_layers - unfreeze_from) / n_layers * 100:.1f}%)')

In [ ]:
# Build the partial-freeze model and apply the freeze strategy.
freeze_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)
n_layers_cfg = freeze_model.config.n_layer  # 24
unfreeze_from_cfg = n_layers_cfg - max(2, n_layers_cfg // 4)  # 18

# Freeze all parameters first
for var in freeze_model.variables:
    var._trainable = False  # type: ignore[attr-defined]
# Unfreeze the last (n_layers - unfreeze_from) transformer blocks
for i in range(unfreeze_from_cfg, n_layers_cfg):
    for var in freeze_model.transformer.h[i].variables:
        var._trainable = True  # type: ignore[attr-defined]
# Unfreeze final layer norm and LM head
for var in freeze_model.transformer.ln_f.variables:
    var._trainable = True  # type: ignore[attr-defined]
for var in freeze_model.lm_head.variables:
    var._trainable = True  # type: ignore[attr-defined]

trainable_freeze = sum(np.prod(v.shape) for v in freeze_model.trainable_variables)
total_freeze = sum(np.prod(v.shape) for v in freeze_model.variables)
print(f'Partial freezing: {trainable_freeze:,} / {total_freeze:,} trainable ({trainable_freeze / total_freeze * 100:.2f}%)')

In [ ]:
# Train the partial-freeze model on a 2-genre slice (same objective as Concept 1: continued pretraining)
optimizer_freeze = tf.keras.optimizers.Adam(learning_rate=1e-4)


@tf.function
def freeze_step(input_ids, attention_mask):
    labels = input_ids[:, 1:]
    with tf.GradientTape() as tape:
        outputs = freeze_model(input_ids, attention_mask=attention_mask, training=True)
        logits = outputs.logits[:, :-1, :]
        pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        loss = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
    grads = tape.gradient(loss, freeze_model.trainable_variables)
    optimizer_freeze.apply_gradients(zip(grads, freeze_model.trainable_variables))
    return loss


freeze_paragraphs = load_corpus_paragraphs(novels=['fantasy', 'cyberpunk'], max_chapters=3)
print(f'Partial-freeze dataset: {len(freeze_paragraphs)} paragraphs')

BATCH_SIZE = 4
MAX_STEPS = 60
LOG_EVERY = 10
loss_history_freeze = []

for step in range(MAX_STEPS):
    idx = (step * BATCH_SIZE) % max(1, len(freeze_paragraphs) - BATCH_SIZE)
    batch = freeze_paragraphs[idx : idx + BATCH_SIZE]
    if not batch:
        continue
    ids, mask = tokenize_batch(batch)
    loss_val = freeze_step(ids, mask)
    loss_history_freeze.append(float(loss_val))
    if (step + 1) % LOG_EVERY == 0:
        print(f'  Step {step + 1:3d}/{MAX_STEPS}  loss={float(loss_val):.4f}')

print(f'\nFinal loss: {loss_history_freeze[-1]:.4f}')

In [ ]:
# Save partial-freeze checkpoint and test
freeze_model.save_pretrained('./checkpoints/partial-freeze')
print('Saved: ./checkpoints/partial-freeze')
print(f'\nPartial-freeze completion: {generate(freeze_model, PROMPT, max_new_tokens=40)}')

### Code Walkthrough: Partial Freezing in TF/Keras

**Why freezing works in TF/Keras:**

In Keras, every variable has a `trainable` flag. When `layer.trainable = False`, all variables belonging to that layer are excluded from `model.trainable_variables`. Since `GradientTape` only computes gradients for variables in `trainable_variables`, frozen layers receive no gradient updates — identical in effect to PyTorch's `param.requires_grad = False`.

**The three-step freeze pattern:**
```python
# 1. Freeze everything
for var in model.variables: var._trainable = False
# 2. Unfreeze last N blocks
for i in range(unfreeze_from, n_layers):
    for var in model.transformer.h[i].variables: var._trainable = True
# 3. Unfreeze output head
for var in model.lm_head.variables: var._trainable = True
```

**TF vs. Keras API note:** The standard `layer.trainable = False` sets the flag at layer level (it cascades to all sublayers). Setting `var._trainable` directly on individual variable objects is the lower-level equivalent, useful when some variables within the same layer need different trainability. Both approaches produce the same `trainable_variables` list.

**Learning rate:** `1e-4` (between full fine-tuning's `5e-5` and parameter-efficient FT's `2e-4`) — partial freezing updates more parameters than the efficient approach but far fewer than full fine-tuning, so the learning rate sits proportionally between them.

---

## Concept 6 (Parameter-Based): TF-Native Parameter-Efficient Fine-Tuning

**Riverside's question for this section:** can we keep one frozen base model on disk and swap in a small, task-specific adapter depending on who's asking — without PEFT?

### TF Note: LoRA is PyTorch-only — What We Do Instead

> **Note: LoRA/PEFT is PyTorch-only.** The PyTorch version of this notebook injects trainable rank-8 A/B matrices alongside each frozen attention weight, giving ~0.5% trainable parameters. PEFT does not have an official TF equivalent that integrates transparently with `TFGPT2LMHeadModel`.
>
> **This Keras version demonstrates "ultra-partial freezing"**: freeze 95%+ of the model (keep only the last 2 transformer blocks + output head trainable). This gives:
> - ~4–5% trainable parameters vs LoRA's <1% (still dramatically cheaper than full fine-tuning or partial freezing)
> - No new architecture needed — just the same freeze-and-unfreeze pattern from Concept 5, applied more aggressively
> - Swappable checkpoints: one frozen base, multiple small fine-tuned copies (the base weights dominate storage, so the marginal cost of each new checkpoint is the delta from the saved full checkpoint)
>
> **What LoRA does that this doesn't:** LoRA injects *new* weight matrices alongside frozen originals, so the adapter is separately serializable without saving the full model. The TF approach here saves the full model each time (less efficient for many adapters, acceptable for Riverside's two use cases).
>
> **To reproduce LoRA in TF:** you would need to subclass `TFGPT2Attention` to inject the A/B linear layers and forward them alongside the frozen base. This is architecturally possible in Keras (it's just subclassing) but adds ~150 lines of boilerplate for a concept already demonstrated in the PyTorch version — see `keras-nlp` for community efforts in this direction.

**The mathematical intuition for why ultra-partial freezing approximates LoRA:**
LoRA trains low-rank updates to a frozen weight: $W + BA$ where $\text{rank}(BA) = r \ll d$. Ultra-partial freezing trains *all* weights in the last 2 blocks — a larger-rank update than LoRA's $r=8$, but still a tiny fraction of the model. The key property is the same: most of the model is unchanged, so catastrophic forgetting risk is low.

In [ ]:
# TF-native parameter-efficient FT: freeze all but the last 2 blocks + output head.
# This is the most aggressive form of partial freezing — ~4-5% trainable parameters.
peft_tf_model = TFGPT2LMHeadModel.from_pretrained(MODEL_NAME)
n_layers_peft = peft_tf_model.config.n_layer  # 24
n_trainable_blocks = 2  # keep only the last 2 blocks trainable
freeze_until = n_layers_peft - n_trainable_blocks  # 22

# Freeze all
for var in peft_tf_model.variables:
    var._trainable = False  # type: ignore[attr-defined]
# Unfreeze only last 2 blocks + output head
for i in range(freeze_until, n_layers_peft):
    for var in peft_tf_model.transformer.h[i].variables:
        var._trainable = True  # type: ignore[attr-defined]
for var in peft_tf_model.lm_head.variables:
    var._trainable = True  # type: ignore[attr-defined]

trainable_peft = sum(np.prod(v.shape) for v in peft_tf_model.trainable_variables)
total_peft = sum(np.prod(v.shape) for v in peft_tf_model.variables)
print(f'TF parameter-efficient FT: {trainable_peft:,} / {total_peft:,} trainable ({trainable_peft / total_peft * 100:.2f}%)')
print(f'This is {n_trainable_blocks} of {n_layers_peft} transformer blocks — our TF approximation of LoRA\'s intent.')

In [ ]:
# Train the TF parameter-efficient model (same continued pretraining objective).
optimizer_peft = tf.keras.optimizers.Adam(learning_rate=2e-4)


@tf.function
def peft_tf_step(input_ids, attention_mask):
    labels = input_ids[:, 1:]
    with tf.GradientTape() as tape:
        outputs = peft_tf_model(input_ids, attention_mask=attention_mask, training=True)
        logits = outputs.logits[:, :-1, :]
        pad_mask = tf.cast(attention_mask[:, 1:], tf.float32)
        token_loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
        loss = tf.reduce_sum(token_loss * pad_mask) / (tf.reduce_sum(pad_mask) + 1e-9)
    grads = tape.gradient(loss, peft_tf_model.trainable_variables)
    optimizer_peft.apply_gradients(zip(grads, peft_tf_model.trainable_variables))
    return loss


peft_paragraphs = load_corpus_paragraphs(novels=['mystery', 'horror', 'literary'], max_chapters=3)
print(f'TF parameter-efficient dataset: {len(peft_paragraphs)} paragraphs')

loss_history_peft = []
for step in range(MAX_STEPS):
    idx = (step * BATCH_SIZE) % max(1, len(peft_paragraphs) - BATCH_SIZE)
    batch = peft_paragraphs[idx : idx + BATCH_SIZE]
    if not batch:
        continue
    ids, mask = tokenize_batch(batch)
    loss_val = peft_tf_step(ids, mask)
    loss_history_peft.append(float(loss_val))
    if (step + 1) % LOG_EVERY == 0:
        print(f'  Step {step + 1:3d}/{MAX_STEPS}  loss={float(loss_val):.4f}')

peft_tf_model.save_pretrained('./checkpoints/peft-tf')
print(f'\nSaved: ./checkpoints/peft-tf  (final loss={loss_history_peft[-1]:.4f})')
print(f'TF-PEFT completion: {generate(peft_tf_model, PROMPT, max_new_tokens=40)}')

### Code Walkthrough: TF-Native Parameter-Efficient Fine-Tuning

**Key observation: learning rate scales with trainability.**

| Technique | Trainable % | Learning rate | Why |
| --- | --- | --- | --- |
| Full fine-tuning | 100% | 5e-5 | Many params — small LR prevents overshooting |
| Partial freezing (25%) | ~21% | 1e-4 | Fewer params — slightly larger LR OK |
| TF parameter-efficient (2 blocks) | ~4-5% | 2e-4 | Very few params — larger LR needed to learn anything |

The pattern: the fewer the trainable parameters, the higher the learning rate needed to achieve a meaningful weight update per step. This is because the loss gradient is spread over fewer dimensions — each trainable parameter needs a larger nudge to produce a similar effect on the loss.

**The training step is structurally identical to Concepts 1 and 5** — same `GradientTape`, same causal LM loss, same padding mask. The only difference is how many variables appear in `peft_tf_model.trainable_variables`. This is the essence of the parameter axis: the *training loop* doesn't change, only the *set of weights that participates*.

---

## TF Note: LoRA and QLoRA (PyTorch-Only — Concept Walkthrough)

> **Note: LoRA/PEFT and QLoRA/bitsandbytes are PyTorch-only.** This section provides the conceptual walkthrough and the code that *would* run on PyTorch — not executed here.

### LoRA: Low-Rank Adaptation

For a weight matrix $W$ (e.g., attention projection), instead of updating $W \rightarrow W + \Delta W$, LoRA:
1. **Freezes** $W$ (no updates ever)
2. **Adds** a low-rank decomposition: $\Delta W = BA$ where $A \in \mathbb{R}^{r \times d}$, $B \in \mathbb{R}^{d \times r}$, and $r \ll d$

**Example for `gpt2-medium`'s attention (d=1024), r=8:**
- Original: $1024 \times 1024 = 1{,}048{,}576$ parameters
- LoRA: $(8 \times 1024) + (1024 \times 8) = 16{,}384$ parameters — 1.56% of original

```python
# PyTorch/PEFT only — not executed in this TF notebook
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=8, lora_alpha=16,
    target_modules=['c_attn'], lora_dropout=0.05, bias='none',
)
lora_model = get_peft_model(base_model_pt, lora_config)
lora_model.print_trainable_parameters()  # trainable params: 2,359,296 || all params: 406,286,336 || trainable%: 0.58%
```

### QLoRA: LoRA on a Quantized Base

QLoRA additionally quantizes the frozen base weights to 4-bit (NF4 format), achieving ~4x memory reduction on top of LoRA's parameter savings.

```python
# PyTorch/PEFT + bitsandbytes only — requires CUDA GPU — not executed here
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)
quantized_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config)
```

**Why this notebook can't run QLoRA:** `bitsandbytes`' 4-bit NF4 kernels require CUDA — no CPU path exists. The concept is fully preserved above; Riverside's laptop simply can't run it.

---

## Quantization for Deployment: TFLite Post-Training Quantization

QLoRA above uses quantization to make *training* fit in less memory. This section addresses a different, practical question: **can the deployed model itself be made smaller and faster to run, independent of how it was trained?**

TensorFlow has a native quantization path via **TFLite** (TensorFlow Lite), which post-training quantizes a SavedModel to int8. This works on CPU (no GPU required) and can produce meaningful size and latency improvements — exactly what Riverside's laptop deployment needs.

| Technique | What's quantized | TF support | Notes |
| --- | --- | --- | --- |
| Dynamic range quantization | Weights to int8; activations float at inference | Full CPU | Fastest to apply; no calibration data |
| Full integer quantization | Weights + activations to int8 | Full CPU/Edge | Needs small representative dataset |
| Float16 quantization | Weights to float16 | CPU/GPU | Good size/quality tradeoff |
| PyTorch dynamic quantization | `torch.quantization.quantize_dynamic` | PyTorch only | The approach used in the PyTorch version of this notebook |

In [ ]:
# TFLite post-training quantization demo.
# Quantizes the non-instruction-full checkpoint to float16 (a safe, well-supported TFLite mode).
# Full int8 quantization for GPT-2 via TFLite requires a representative dataset and
# specific TFLite conversion configuration — float16 is the practical starting point.
import os, math, io

if non_instruct_ckpt is not None:
    # Export the model to TF SavedModel format (required for TFLite conversion)
    saved_model_path = './checkpoints/non-instruction-full-savedmodel'
    non_instruct_ckpt.save_pretrained(saved_model_path, saved_model=True)
    print(f'Saved TF SavedModel to: {saved_model_path}')

    # Measure fp32 checkpoint size on disk
    def dir_size_mb(path):
        total = 0
        for dirpath, _, filenames in os.walk(path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if os.path.isfile(fp):
                    total += os.path.getsize(fp)
        return total / 1e6

    fp32_size = dir_size_mb('./checkpoints/non-instruction-full')
    print(f'fp32 checkpoint size: {fp32_size:.1f} MB')
    print('TFLite conversion ready (to run full conversion, use tf.lite.TFLiteConverter).')
    print('Demonstrating conceptual size comparison instead (avoids TFLite shape-inference overhead on CPU):')
    # Theoretical float16 size: 50% of fp32
    print(f'  fp32  : {fp32_size:.1f} MB')
    print(f'  fp16  : ~{fp32_size * 0.5:.1f} MB  (theoretical 2x reduction via TFLite float16)')
    print(f'  int8  : ~{fp32_size * 0.25:.1f} MB  (theoretical 4x reduction via TFLite full int8)')
else:
    print('non_instruct_ckpt not loaded — run Part 1 first.')

---

## Visual Comparison: Parameter Counts Across All Techniques

The three techniques trained in this notebook use the same continued-pretraining objective — so parameter count is the only variable. Let's compare the real numbers.

In [ ]:
# Real parameter counts from the models trained in this notebook
full_ft_params = sum(np.prod(v.shape) for v in base_model.trainable_variables)  # 100%
partial_ft_params = sum(np.prod(v.shape) for v in freeze_model.trainable_variables)
peft_tf_params = sum(np.prod(v.shape) for v in peft_tf_model.trainable_variables)
total_p = sum(np.prod(v.shape) for v in base_model.variables)

techniques_names = ['Full\nFine-Tuning', 'Partial\nFreezing', 'TF-Native\nPEFT']
param_counts = [full_ft_params, partial_ft_params, peft_tf_params]
param_pcts = [c / total_p * 100 for c in param_counts]
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Parameter Count Comparison — {MODEL_NAME} ({total_p / 1e6:.0f}M total params)', fontsize=12, fontweight='bold')

# Absolute counts (log scale)
bars = ax1.bar(techniques_names, param_counts, color=colors_bar, alpha=0.85, edgecolor='black')
ax1.set_yscale('log')
ax1.set_ylabel('Trainable Parameters (log scale)', fontsize=10)
ax1.set_title('Absolute Trainable Parameters', fontweight='bold')
for bar, count, pct in zip(bars, param_counts, param_pcts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.2,
             f'{count:,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax1.grid(alpha=0.3, axis='y')

# Memory comparison (approximate optimizer state = 3x weights for Adam: weights + m + v)
bytes_per_param_weights = 4  # float32
bytes_per_param_optimizer = 8  # Adam m + v
mem_gb = [(c * (bytes_per_param_weights + bytes_per_param_optimizer)) / 1e9 for c in param_counts]
ax2.bar(techniques_names, mem_gb, color=colors_bar, alpha=0.85, edgecolor='black')
ax2.set_ylabel('Training Memory Estimate (GB)', fontsize=10)
ax2.set_title('Approximate Optimizer Memory Requirement\n(weights + Adam state)', fontweight='bold')
for i, (mem, pct) in enumerate(zip(mem_gb, param_pcts)):
    ax2.text(i, mem + 0.01, f'{mem:.3f} GB', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'\n{"=" * 70}')
print(f'{"Technique":<22} {"Trainable":<15} {"%":<8} {"Memory Est."}')
print(f'{"=" * 70}')
for name, cnt, pct, mem in zip(techniques_names, param_counts, param_pcts, mem_gb):
    print(f'{name.replace(chr(10), " "):<22} {cnt:>13,}  {pct:>7.2f}%  ~{mem:.3f} GB')
print(f'{"=" * 70}')

---

## End of Part 2: Five Checkpoints, One Open Question

Two more checkpoints now exist on disk, joining Part 1's three:

| Checkpoint | What it is |
| --- | --- |
| `./checkpoints/partial-freeze` | Partial fine-tuning / layer freezing, continued pretraining (Concept 5) |
| `./checkpoints/peft-tf` | TF-native parameter-efficient FT (ultra-partial freeze, 2 blocks), continued pretraining (Concept 6) |

Between Part 1 and Part 2, Riverside now has **five trained checkpoints** covering four of the nine data × parameter combinations. What's still missing: **a real, side-by-side, quantitative comparison of all five** — which one actually deserves to be deployed?

Continue to **[Part 3: Comparison & Decision](03-llm-finetuning-comparison-and-decision.ipynb)**, which reloads all five checkpoints from disk and puts them head-to-head: qualitative side-by-side generations, token-probability analysis, held-out perplexity, the full data × parameter combination grid, an ablation study, and the final call on what Riverside House actually ships.

### Key Insights from Part 2

- **Full fine-tuning** is already run in Part 1 — the `non-instruction-full` checkpoint is the controlled baseline.
- **Partial freezing** trains ~21% of parameters at `1e-4` LR; the frozen early blocks preserve general English while the trainable late blocks absorb the new domain data.
- **TF-native PEFT** trains ~4-5% of parameters at `2e-4` LR; the even smaller trainable set means stronger generalization pressure but less capacity to absorb new information.
- **LoRA** (the PyTorch production standard) would achieve <1% trainable parameters using injected low-rank matrices — conceptually the same intent as ultra-partial freezing, but via adapter injection rather than layer selection. Use the PyTorch version of this notebook to run that.
- **QLoRA** combines LoRA with 4-bit quantization of the frozen base — GPU-only (`bitsandbytes`), explained but not run in either version.